# Aula 13 - Memória do Agente com Grafos de Conhecimento Cognee


## Configuração

Este notebook demonstra como construir um **assistente de programação** inteligente com memória persistente usando grafos de conhecimento [**Cognee**](https://www.cognee.ai/) e o **Microsoft Agent Framework** (MAF).

O Cognee transforma texto não estruturado em um grafo de conhecimento estruturado e consultável sustentado por embeddings vetoriais — dando ao seu agente uma memória de longo prazo rica e consciente das relações.

### O que irá aprender
1. **Construir Grafos de Conhecimento**: Transformar perfis de programadores e melhores práticas em conhecimento estruturado e consultável.
2. **Integrar Cognee com MAF**: Usar funções `@tool` para permitir que um agente MAF consulte o grafo de conhecimento do Cognee.
3. **Conversas com Consciência de Sessão**: Manter o contexto em várias perguntas na mesma sessão.
4. **Memória de Longo Prazo**: Persistir conhecimento importante entre sessões e recuperá-lo em novas conversas.

### Pré-requisitos
- Python 3.9+
- Redis a correr localmente (`docker run -d -p 6379:6379 redis`) para gestão de sessões
- Uma chave de API LLM (por exemplo, OpenAI) — configurar `LLM_API_KEY` no `.env`
- `CACHING=true` no `.env` (requerido para sessões Cognee)
- Um projeto Azure AI Foundry com um modelo de chat implementado
- `AZURE_AI_PROJECT_ENDPOINT` e `AZURE_AI_MODEL_DEPLOYMENT_NAME` no `.env`
- Azure CLI autenticado (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipos de Memória do Agente

Este notebook explora os mesmos três tipos de memória do notebook principal da Aula 13, mas usa Cognee como a memória de longo prazo:

| Tipo de Memória | Mecanismo | Duração |
|---|---|---|
| **Ativa** | `agent.create_session()` (MAF) | Conversa única |
| **Curto-prazo** | Cache de sessão Cognee (Redis) | Sessão única |
| **Longo-prazo** | Grafo de conhecimento + vetores Cognee | Permanente |

### Arquitetura de Memória do Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Preparar Armazenamento Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Parte 1 — Construção da Base de Conhecimento

Ingerimos três tipos de dados para criar uma base de conhecimento abrangente para o nosso assistente de programação:

1. **Perfil do Desenvolvedor** — expertise pessoal e formação técnica  
2. **Melhores Práticas de Python** — o Zen do Python com orientações práticas  
3. **Conversas Históricas** — sessões anteriores de perguntas e respostas entre desenvolvedores e assistentes de IA


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualizar o Grafo de Conhecimento

O Cognee pode gerar uma visualização HTML interativa das entidades e relações que extraiu.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enriquecer a Memória com Memify

`memify()` analisa o grafo de conhecimento e gera regras inteligentes — identificando padrões, boas práticas e relações entre conceitos.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Parte 2 — Agente MAF com Ferramentas Cognee

Agora criamos um agente MAF que pode consultar o grafo de conhecimento da Cognee através das funções `@tool`. Isto permite ao agente aproveitar todo o poder da pesquisa semântica consciente do grafo enquanto mantém o contexto de conversação através das sessões.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Memória de Trabalho com Sessões

O `AgentSession` (criado através de `agent.create_session()`) fornece memória de trabalho dentro de uma sessão. O agente pode referir-se a mensagens anteriores enquanto também consulta o gráfico de conhecimento de longo prazo da Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nova Sessão — A Memória de Longo Prazo Persiste

Iniciar uma nova sessão limpa a memória de trabalho, mas o grafo de conhecimento Cognee continua disponível. O agente pode recuperar o mesmo conhecimento de longo prazo numa conversa completamente nova.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumo

Neste notebook construíste um assistente de programação que combina a **memória de trabalho do MAF** (`agent.create_session()`) com o **grafo de conhecimento a longo prazo da Cognee**.

### O que aprendeste
1. **Construção de grafo de conhecimento**: A Cognee processa texto não estruturado e constrói um grafo + memória vetorial.
2. **Enriquecimento do grafo com memify**: Factos derivados e relações mais ricas em cima do teu grafo existente.
3. **Integração MAF + Cognee**: As funções `@tool` permitem que um agente MAF consulte o grafo da Cognee de forma natural.
4. **Memória de trabalho + memória a longo prazo**: `AgentSession` (via `agent.create_session()`) fornece contexto por sessão enquanto a Cognee oferece conhecimento persistente.
5. **Pesquisa filtrada com NodeSets**: Direciona subconjuntos específicos do grafo de conhecimento (ex. apenas princípios).

### Principais conclusões
- **Cognee** transforma texto cru em memória estruturada e consciente das relações — mais poderosa do que uma simples loja de vetores.
- As funções **`@tool`** fazem a ligação limpa entre agentes MAF e sistemas externos de conhecimento.
- `AgentSession` (via `agent.create_session()`) mantém o contexto por conversa separado do conhecimento duradouro.
- O mesmo grafo de conhecimento serve múltiplas sessões e agentes.

### Aplicações no mundo real
- **Copilotos para desenvolvedores**: Revisão de código, análise de incidentes, assistentes de arquitetura
- **Copilotos para atendimento ao cliente**: Agentes de suporte baseados em documentação de produto, FAQs e notas de CRM
- **Copilotos internos especialistas**: Assistentes de políticas, legais ou de segurança que raciocinam sobre diretrizes
- **Camadas unificadas de dados**: Combinar dados estruturados e não estruturados num único grafo consultável

### Próximos passos
- Experimentar consciência temporal na Cognee
- Definir uma ontologia OWL para qualidade do grafo específica de domínio
- Adicionar ciclos de feedback de utilizador para melhorar a recuperação ao longo do tempo
- Escalar para sistemas multi-agente que partilham a mesma camada de memória da Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos por garantir a precisão, tenha em atenção que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se a tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erradas decorrentes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
